In [20]:
import os
import pandas as pd
import numpy as np

In [21]:
daphnet_config = {
    "label_col": 10,
    "fog_value": 2,
    "nonfog_value": 1,
    "sensor_columns": {"ankle": [1, 2, 3], "thigh": [4, 5, 6], "trunk": [7, 8, 9]},
    "column_type": "index",
}

tlvmc_config = {
    "label_col": "fog",
    "fog_value": 1,
    "nonfog_value": 0,
    "sensor_columns": {"lower_back": ["AccV", "AccML", "AccAP"]},
    "column_type": "name",
}

In [22]:
def load_dataset(folder_path, file_type="txt"):
    data = []

    for file in os.listdir(folder_path):
        if file.endswith(file_type):
            path = os.path.join(folder_path, file)

            if file_type == "txt":
                df = pd.read_csv(path, delim_whitespace=True, header=None)
            else:
                df = pd.read_csv(path)

            df["file"] = file
            data.append(df)

    return pd.concat(data, ignore_index=True)

In [23]:
def create_tlvmc_fog(df):
    df["fog"] = (
        ((df["StartHesitation"] == 1) | (df["Turn"] == 1) | (df["Walking"] == 1))
        & (df["Valid"] == 1)
        & (df["Task"] == 1)
    ).astype(int)

    return df

In [24]:
def distribution_analysis(df, config):
    label_col = config["label_col"]
    fog_val = config["fog_value"]

    total = len(df)
    fog_count = (df[label_col] == fog_val).sum()

    return {
        "total_samples": total,
        "fog_samples": fog_count,
        "fog_percent": (fog_count / total) * 100,
    }

In [25]:
def temporal_analysis(df, config):
    label_col = config["label_col"]
    fog_val = config["fog_value"]

    durations = []

    for file, group in df.groupby("file"):
        labels = group[label_col].values

        fog_indices = np.where(labels == fog_val)[0]
        segments = np.split(fog_indices, np.where(np.diff(fog_indices) != 1)[0] + 1)

        durations.extend([len(seg) for seg in segments if len(seg) > 0])

    return {
        "num_episodes": len(durations),
        "mean_duration": np.mean(durations),
        "max_duration": np.max(durations),
        "min_duration": np.min(durations),
    }

In [26]:
def compute_energy(signal):
    if len(signal) == 0:
        return np.nan
    return np.mean(signal**2)


def zero_crossing_rate(signal):
    if len(signal) == 0:
        return np.nan
    return np.mean(np.diff(np.sign(signal)) != 0)

In [27]:
def signal_behavior_analysis(df, config):
    label_col = config["label_col"]
    fog_val = config["fog_value"]
    nonfog_val = config["nonfog_value"]

    results = []

    for file, group in df.groupby("file"):
        for sensor, cols in config["sensor_columns"].items():

            for i, col in enumerate(cols):

                # Select column properly
                if config["column_type"] == "index":
                    signal = group.iloc[:, col]
                else:
                    signal = group[col]

                fog_signal = signal[group[label_col] == fog_val]
                nonfog_signal = signal[group[label_col] == nonfog_val]

                results.append(
                    {
                        "file": file,
                        "sensor": sensor,
                        "axis": i,
                        # STD
                        "fog_std": fog_signal.std(),
                        "nonfog_std": nonfog_signal.std(),
                        # ENERGY
                        "fog_energy": compute_energy(fog_signal),
                        "nonfog_energy": compute_energy(nonfog_signal),
                        # ZCR
                        "fog_zcr": zero_crossing_rate(fog_signal),
                        "nonfog_zcr": zero_crossing_rate(nonfog_signal),
                    }
                )

    return pd.DataFrame(results)

In [28]:
def consistency_analysis(df):
    df["std_consistency"] = df["fog_std"] > df["nonfog_std"]
    df["energy_consistency"] = df["fog_energy"] > df["nonfog_energy"]
    df["zcr_consistency"] = df["fog_zcr"] > df["nonfog_zcr"]

    summary = df.groupby(["sensor", "axis"]).agg(
        {
            "std_consistency": "mean",
            "energy_consistency": "mean",
            "zcr_consistency": "mean",
        }
    )

    return summary

In [29]:
def event_contribution(df):
    total_fog = df["fog"].sum()

    return {
        "StartHesitation": df["StartHesitation"].sum() / total_fog,
        "Turn": df["Turn"].sum() / total_fog,
        "Walking": df["Walking"].sum() / total_fog,
    }

In [30]:
def aggregate_dataset_results(per_file_df):

    agg_df = (
        per_file_df.groupby(["sensor", "axis"])
        .agg(
            {
                "fog_std": "mean",
                "nonfog_std": "mean",
                "fog_energy": "mean",
                "nonfog_energy": "mean",
                "fog_zcr": "mean",
                "nonfog_zcr": "mean",
            }
        )
        .reset_index()
    )

    return agg_df

In [31]:
def add_feature_differences(df):
    df["std_diff"] = df["fog_std"] - df["nonfog_std"]
    df["energy_diff"] = df["fog_energy"] - df["nonfog_energy"]
    df["zcr_diff"] = df["fog_zcr"] - df["nonfog_zcr"]

    return df

In [32]:
DAPHNET_PATH = r"../data/raw/DAPHNET/dataset"
df_daphnet = load_dataset(DAPHNET_PATH, "txt")

C:\Users\Tanay Yeole\AppData\Local\Temp\ipykernel_34040\2684670083.py:9: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  df = pd.read_csv(path, delim_whitespace=True, header=None)
C:\Users\Tanay Yeole\AppData\Local\Temp\ipykernel_34040\2684670083.py:9: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  df = pd.read_csv(path, delim_whitespace=True, header=None)
C:\Users\Tanay Yeole\AppData\Local\Temp\ipykernel_34040\2684670083.py:9: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  df = pd.read_csv(path, delim_whitespace=True, header=None)
C:\Users\Tanay Yeole\AppData\Local\Temp\ipykernel_34040\2684670083.py:9: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a

In [33]:
distribution_analysis(df_daphnet, daphnet_config)

{'total_samples': 1917887,
 'fog_samples': np.int64(110785),
 'fog_percent': np.float64(5.776409141935892)}

In [34]:
temporal_analysis(df_daphnet, daphnet_config)

{'num_episodes': 237,
 'mean_duration': np.float64(467.44725738396625),
 'max_duration': np.int64(2594),
 'min_duration': np.int64(28)}

In [35]:
std_df_daphnet = signal_behavior_analysis(df_daphnet, daphnet_config)
consistency_analysis(std_df_daphnet)

std_consistency  energy_consistency  zcr_consistency
sensor axis                                                      
ankle  0            0.411765            0.470588         0.705882
       1            0.470588            0.294118         0.764706
       2            0.352941            0.294118         0.705882
thigh  0            0.352941            0.176471         0.705882
       1            0.000000            0.764706         0.470588
       2            0.470588            0.352941         0.764706
trunk  0            0.411765            0.470588         0.588235
       1            0.411765            0.588235         0.117647
       2            0.176471            0.235294         0.647059

In [36]:
per_file_daphnet = signal_behavior_analysis(df_daphnet, daphnet_config)

dataset_daphnet = aggregate_dataset_results(per_file_daphnet)
dataset_daphnet = add_feature_differences(dataset_daphnet)

consistency_daphnet = consistency_analysis(per_file_daphnet)
consistency_daphnet

std_consistency  energy_consistency  zcr_consistency
sensor axis                                                      
ankle  0            0.411765            0.470588         0.705882
       1            0.470588            0.294118         0.764706
       2            0.352941            0.294118         0.705882
thigh  0            0.352941            0.176471         0.705882
       1            0.000000            0.764706         0.470588
       2            0.470588            0.352941         0.764706
trunk  0            0.411765            0.470588         0.588235
       1            0.411765            0.588235         0.117647
       2            0.176471            0.235294         0.647059

In [37]:
TLVMC_PATH = r"D:\tlvmc-parkinsons-freezing-gait-prediction\train\defog"
df_tlvmc = load_dataset(TLVMC_PATH, "csv")

df_tlvmc = create_tlvmc_fog(df_tlvmc)

In [38]:
distribution_analysis(df_tlvmc, tlvmc_config)

{'total_samples': 13525702,
 'fog_samples': np.int64(685847),
 'fog_percent': np.float64(5.070694297419831)}

In [39]:
temporal_analysis(df_tlvmc, tlvmc_config)

{'num_episodes': 1292,
 'mean_duration': np.float64(530.8413312693499),
 'max_duration': np.int64(14456),
 'min_duration': np.int64(12)}

In [40]:
std_df_tlvmc = signal_behavior_analysis(df_tlvmc, tlvmc_config)
consistency_analysis(std_df_tlvmc)

std_consistency  energy_consistency  zcr_consistency
sensor     axis                                                      
lower_back 0            0.527473            0.318681         0.021978
           1            0.846154            0.780220         0.945055
           2            0.021978            0.692308         0.142857

In [41]:
event_contribution(df_tlvmc)

{'StartHesitation': np.float64(0.0007290255698428367),
 'Turn': np.float64(0.8565846318493775),
 'Walking': np.float64(0.14364574023069285)}

In [42]:
per_file_tlvmc = signal_behavior_analysis(df_tlvmc, tlvmc_config)
dataset_tlvmc = aggregate_dataset_results(per_file_tlvmc)
dataset_tlvmc = add_feature_differences(dataset_tlvmc)

consistency_tlvmc = consistency_analysis(per_file_tlvmc)
consistency_tlvmc

std_consistency  energy_consistency  zcr_consistency
sensor     axis                                                      
lower_back 0            0.527473            0.318681         0.021978
           1            0.846154            0.780220         0.945055
           2            0.021978            0.692308         0.142857

### Resampling


In [43]:
def resample_dataset(df, config, original_freq, target_freq):

    factor = target_freq / original_freq

    sensor_cols = []
    for cols in config["sensor_columns"].values():
        sensor_cols.extend(cols)

    label_col = config["label_col"]

    resampled_data = []

    for file, group in df.groupby("file"):

        group = group.reset_index(drop=True)

        old_len = len(group)
        new_len = int(old_len * factor)

        # Create new indices
        new_indices = np.linspace(0, old_len - 1, new_len)
        new_indices_int = new_indices.astype(int)

        # Resample signals (nearest for speed)
        resampled_group = group.iloc[new_indices_int].copy()

        resampled_group["file"] = file

        resampled_data.append(resampled_group)

    return pd.concat(resampled_data, ignore_index=True)

### Windowing


In [44]:
def create_windows(df, config, window_size=256, step_size=128):

    label_col = config["label_col"]
    fog_val = config["fog_value"]

    windows = []
    labels = []

    sensor_cols = []
    for cols in config["sensor_columns"].values():
        sensor_cols.extend(cols)

    for file, group in df.groupby("file"):

        group = group.reset_index(drop=True)

        # Extract signals
        if config["column_type"] == "index":
            signals = group.iloc[:, sensor_cols].values
        else:
            signals = group[sensor_cols].values

        label_array = group[label_col].values

        for start in range(0, len(group) - window_size, step_size):

            end = start + window_size

            window = signals[start:end]
            window_labels = label_array[start:end]

            # ANY-FoG labeling
            label = 1 if np.any(window_labels == fog_val) else 0

            windows.append(window)
            labels.append(label)

    return np.array(windows), np.array(labels)

In [45]:
def window_distribution(y):
    total = len(y)
    fog = np.sum(y == 1)

    return {
        "total_windows": total,
        "fog_windows": fog,
        "fog_percent": (fog / total) * 100,
    }

In [46]:
# If DAPHNet is already ~64Hz, skip resampling
df_daphnet_resampled = df_daphnet.copy()

X_daphnet, y_daphnet = create_windows(df_daphnet_resampled, daphnet_config)

print(window_distribution(y_daphnet))

{'total_windows': 14958, 'fog_windows': np.int64(1316), 'fog_percent': np.float64(8.797967642732987)}


In [47]:
df_tlvmc_resampled = resample_dataset(
    df_tlvmc, tlvmc_config, original_freq=100, target_freq=64  # adjust if needed
)

X_tlvmc, y_tlvmc = create_windows(df_tlvmc_resampled, tlvmc_config)

print(window_distribution(y_tlvmc))

{'total_windows': 67495, 'fog_windows': np.int64(5507), 'fog_percent': np.float64(8.159122897992443)}


In [48]:
print(X_daphnet.shape)
print(X_tlvmc.shape)

(14958, 256, 9)
(67495, 256, 3)


In [49]:
print(window_distribution(y_daphnet))
print(window_distribution(y_tlvmc))

{'total_windows': 14958, 'fog_windows': np.int64(1316), 'fog_percent': np.float64(8.797967642732987)}
{'total_windows': 67495, 'fog_windows': np.int64(5507), 'fog_percent': np.float64(8.159122897992443)}
